In [1]:
from dataclasses import dataclass
import torch
import torch.nn as nn
import math




GPTConfig - The dataclas that governs the size of the model

vocab_size - vocabulaty size, how many unique words model knows

block_size - Context window, how far the model can see into the past

n_layer - model depth, how many layers the model has

n_head - model width, how many parrallel conversations the attention machanism has

n_embed - Embedding dimension, The size of the vectors representing each token

In [2]:
@dataclass
class GPTConfig:
  vocab_size: int
  block_size: int
  n_layer: int = 12
  n_head: int = 12
  n_embed: int = 768
  dorpout: float = 0.1

# Embedding layers

It has requires_grad = True, so the model will learn the best embedding for each token during training.

Words with simmilar semantic meanings will be colose to one and other in this vector space.

The problem currently is that this represenatation is static, no matter the words surrounding it.

It needs to be dynamically adjusted based on it's conetxt.

We need positional embeddings.

In [3]:
vocab_size = 20
n_embed = 3 # number of embedding dimensions, called "semantic space"

token_embedding_table = nn.Embedding(vocab_size, n_embed)

print(f"Shape of the embedding = {token_embedding_table.weight.shape}")
print(f"Contents of the embedding:\n{token_embedding_table.weight}")

Shape of the embedding = torch.Size([20, 3])
Contents of the embedding:
Parameter containing:
tensor([[-1.8074, -0.8823, -2.1930],
        [ 1.0692, -0.6226, -0.0367],
        [ 0.6081,  0.2102, -1.6453],
        [ 0.4922, -0.3796, -1.0840],
        [-0.0391,  1.5227, -0.6390],
        [ 0.9037,  1.1409,  2.2220],
        [ 1.8530,  0.0718,  0.7567],
        [ 0.8813, -0.9097,  1.2140],
        [ 0.5408,  0.9942,  0.9349],
        [ 0.8824,  1.9481, -0.7019],
        [ 0.3319, -0.0258, -0.8705],
        [-0.6799, -1.4292,  1.0497],
        [ 0.5444, -3.1362, -1.3346],
        [-0.2651, -0.7809,  0.7093],
        [ 0.7064, -1.1980, -1.6072],
        [ 0.5776, -1.4084, -2.1789],
        [-0.5410,  0.3675, -1.1282],
        [-0.4143,  0.6481, -1.4614],
        [-0.8708, -0.0753,  0.0299],
        [-0.8555, -0.5020,  1.2519]], requires_grad=True)


# Positional embedding

It works because they:

1. They exist in the same space
2. Adding them creates a unique vector in that space
3. Workd 'Cat' at position 1, is different than 'Cat' at pisition 11

Same as numpy, pytorch uses broadcasting

In [4]:
batch, time, channels = 3, 8, 3 # time = sequance length, channels = enbedding dimensions
vocab_size = 20
block_size = 10

token_embedding_table = nn.Embedding(vocab_size, channels)

# Position embedding, it's vocab is the block size
position_embedding_table = nn.Embedding(block_size, channels)

input_data = torch.randint(0, vocab_size, (batch, time))

#1 - getting the token embeddings
token_embeddings = token_embedding_table(input_data)

#2 - Getting the position tokens for our current sequence length
input_positions = torch.arange(0, time, dtype=torch.long)
positional_embeddings = position_embedding_table(input_positions)

#3 - Adding the together to get the result
x = token_embeddings + positional_embeddings

print(f"Result dimensions = {x.shape}\nToken Embedding dim = {token_embeddings.shape}\nPositional emb. shape = {positional_embeddings.shape}")
print(f"\n{'-__' * 40}\n\nReuslt = {x[:1]}\n\n{'-__' * 40}\n\nToken Emb. = {token_embeddings[:1]}")

Result dimensions = torch.Size([3, 8, 3])
Token Embedding dim = torch.Size([3, 8, 3])
Positional emb. shape = torch.Size([8, 3])

-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__

Reuslt = tensor([[[ 0.3841,  0.5271, -0.9089],
         [ 0.9225, -1.3533,  0.9660],
         [ 3.6452,  0.8538, -0.8132],
         [-0.0604,  0.8899, -1.5861],
         [-2.9294,  1.2458, -0.3268],
         [-0.6583,  1.8698, -0.9553],
         [-0.8296, -0.1080,  0.6384],
         [-0.7214,  0.0460,  0.3073]]], grad_fn=<SliceBackward0>)

-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__-__

Token Emb. = tensor([[[ 0.2614,  0.2735, -0.3518],
         [-0.2657, -0.7485, -1.8948],
         [ 0.8945,  0.8958,  0.4103],
         [-0.4233,  0.0582, -0.6093],
         [-0.9777,  0.1436, -0.4041],
         [-0.9777,  0.1436, -0.4041],
         [-0.4233,  0.0582, -0.6093],
     

We got the positional embeddings, but the model still does not care about the word's/tokens neighbours, just it's position. To change that we need:

# Attention:

Causal Self-Attention: $$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V $$

Each word is represented with 3 distinct vectors:

* Q: query, the word's search query, what it's looking for
* K: key, the words 'label' / 'keyword', What the word is
* V: value, the words payload, the information it offers.

V is the packaged consumable infor, as compared tro x whitch is the raw complete information in it's entirety. So V is just a learned transformation of x, so V = x @ W_v, and x is the output of the embedding layers.

The process has 3 steps:

1. Scoring
2. Normalizing
3. Aggregating


Self attention is a **Cumminication Mechanism** that allows every word to dynamically pull in information from it's context to refine it's own meaning.

In [5]:
batch, time, channels = 1, 4, 2 # batch(onesentence), time(4 words), channels(numb of emb dimentions)

x = torch.tensor(
    [[[0.1, 0.1], # A
     [1.0, 0.2], # Crane
     [0.1, 0.9], # ate
     [0.8, 0.0]]] # fish
).float()

In [6]:
# learnable components
q_projection = nn.Linear(channels, channels, bias=False)
k_projection = nn.Linear(channels, channels, bias=False)
v_projection = nn.Linear(channels, channels, bias=False)

# Manuallt setting the weights. the seed so for the reproducibility sake
torch.manual_seed(42)
q_projection.weight.data = torch.randn(channels, channels)
k_projection.weight.data = torch.randn(channels, channels)
v_projection.weight.data = torch.randn(channels, channels)

# perform the projection
q = q_projection(x)
k = k_projection(x)
v = v_projection(x)

In [7]:
# with torch.no_grad():
#     q_projection = nn.Linear(channels, channels, bias=False)
#     k_projection = nn.Linear(channels, channels, bias=False)
#     v_projection = nn.Linear(channels, channels, bias=False)
#     # Force the projections to do nothing (Identity)
#     q_projection.weight.copy_(torch.eye(channels))
#     k_projection.weight.copy_(torch.eye(channels))
#     v_projection.weight.copy_(torch.eye(channels))

#     q = q_projection(x)
#     k = k_projection(x)
#     v = v_projection(x)

#     scores = q @ k.transpose(-2 ,-1)

#     print(scores.shape, scores)

#     d_k = k.size(-1)
#     scaled_scores = scores / math.sqrt(d_k)
#     attention_weights = torch.nn.functional.softmax(scaled_scores * 10, dim=-1)

#     output = attention_weights @ v
#     print(output)

## Calculating attention Scores

Core of the communication. We're computing the dot product of every's token qwuery, with every other token's key.

To matmul q with k we need to transpose k's last 2 dimentions.

k origian dims = 1, 4, 2

k.transpose = 1, 2 ,4

q * k = 1 4 4

In [8]:
scores = q @ k.transpose(-2 ,-1)

In [9]:
scores.shape, scores

(torch.Size([1, 4, 4]),
 tensor([[[ 0.0012,  0.0427, -0.0295,  0.0403],
          [-0.0034,  0.1632, -0.2006,  0.1700],
          [ 0.0166,  0.3065, -0.1234,  0.2732],
          [-0.0058,  0.0778, -0.1417,  0.0894]]], grad_fn=<UnsafeViewBackward0>))

## Scale an softmax

scaling for numerical stability

softmax to turn raw weight, into weights that sum to 1, so precentages (kinda)

In [10]:
d_k = k.size(-1)
scaled_scores = scores / math.sqrt(d_k)
attention_weights = torch.nn.functional.softmax(scaled_scores, dim=-1)

In [11]:
output = attention_weights @ v
output

tensor([[[0.3131, 0.5096],
         [0.3198, 0.5047],
         [0.3250, 0.5104],
         [0.3157, 0.5055]]], grad_fn=<UnsafeViewBackward0>)

# Putting it all together into a class

In [12]:
class SingleHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.c_attn = nn.Linear(config.n_embed, 3 * config.n_embed, bias=False)

  def forward(self, x):
    batch, time, channels = x.size()

    # Get the query, key, value vector form a single projection and slit them
    q, k, v= self.c_attn(x).split(channels, dim=2)
    # q, k, v = qkv.split(C, dim=2)

    # Calculate attention weights
    scaled_scores = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
    attention_weights = torch.nn.functional.softmax(scaled_scores, dim=-1)

    output = attention_weights @ v

    return output

Autoregressive Model - Model that generates one word at a time.

The model can only see into the past

We need to use a cause mas, we mask out futore positions by setting thier score to -inf.

We use negative because the exp(-inf) = 0

In [13]:
# Creating the mask
T = 4
mask = torch.tril(torch.ones(T, T))
mask

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [14]:
masked_scores = scaled_scores.masked_fill(mask == 0, float('-inf'))
masked_scores

tensor([[[ 0.0009,    -inf,    -inf,    -inf],
         [-0.0024,  0.1154,    -inf,    -inf],
         [ 0.0118,  0.2168, -0.0873,    -inf],
         [-0.0041,  0.0550, -0.1002,  0.0632]]], grad_fn=<MaskedFillBackward0>)

upper right triangle is now all zeros. The words can not attend to future words anymore.

In [15]:
attention_weights = torch.nn.functional.softmax(masked_scores, dim=-1)
attention_weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.4706, 0.5294, 0.0000, 0.0000],
         [0.3192, 0.3918, 0.2891, 0.0000],
         [0.2476, 0.2627, 0.2249, 0.2648]]], grad_fn=<SoftmaxBackward0>)

A buffer is a tensor that:
* is part of the model's state, just like wieghts
* It gets saved with the model ('state_dict')
* it moves to GPU with 'to.(device)'
* **it is not a parameter, so it is not updated by the optimizer**

In [16]:
class CausalSelfAttention:
  def __init__(self, config):
    super().__init__()
    self.c_attn = nn.Linear(config.n_embed, 3 * config.n_embed, bias=False)

    # we register the mask as a "buffer"
    self.register_buffer(
        "bias",
        torch.tril(torch.ones(config.block_size, config.block_size))
        .view(1, 1, config.block_size, config.block_size)
    )

    def forward(self, x):
      batch, time, channels = x.size()

      # Get the query, key, value vector form a single projection and slit them
      q, k, v= self.c_attn(x).split(channels, dim=2)
      # q, k, v = qkv.split(C, dim=2)

      # Calculate attention weights
      scaled_scores = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
      # Slicing the stored mask to the correct sequance length time
      scaled_scores = scaled_scores.masked_fill(self.bias[:, :, :time, :time] == 0, float('-inf'))
      attention_weights = torch.nn.functional.softmax(scaled_scores, dim=-1)

      output = attention_weights @ v

      return output

To make attentions really powerfull, we allow it to have maby 'conversations' at the same time. We add multiple heads to the attention, each head might specilize in different things, like head one specializes in syntax, head two specializes in semantics and so on.

It follows 3 steps:
1.  Split
2.  Attend
3.  Merge

Let's use the attention form GPT2

Embedding Dim = 768

Num of heads = 12

DImension per head = Emb.Dim / nHeads = 64

In [17]:
batch, time, channels = 1, 4, GPTConfig.n_embed

head_dim = GPTConfig.n_embed // GPTConfig.n_head

q = torch.randn(batch, time, channels)
k = torch.randn(batch, time, channels)
v = torch.randn(batch, time, channels)

In [18]:
q_reshaped = q.view(batch, time, GPTConfig.n_head, head_dim)
print(f"After reshaping with .view(): {q_reshaped.shape}")

After reshaping with .view(): torch.Size([1, 4, 12, 64])


We need to bring the number of heads to the front. It allows pytorch to process the heads in parallel.

they get treated by another batch dimention by the pytorch broadcasting.

The attention mathc will happen 12 times faster and independantly

In [19]:
q_final = q_reshaped.transpose(1, -2)
q_final.shape

torch.Size([1, 12, 4, 64])

Run attention in parallel

In [20]:
k_final = k.view(batch, time, GPTConfig.n_head, head_dim).transpose(1, 2)
v_final = k.view(batch, time, GPTConfig.n_head, head_dim).transpose(1, 2)


# Attention Calculation (For all the head at once)
# (B, n_heads, T, head_dim) @ (B, n_heads, head_dim, T) -> (B, n_heads, T, T)
scaled_scores = (q_final @ k_final.transpose(-2, -1)) / math.sqrt(head_dim)
attention_weights = torch.nn.functional.softmax(scaled_scores, dim=-1)

# (B, n_heads, T, T) @ (B, n_heads, T, head_dim)
output_per_head = attention_weights @ v_final
print(f"Shape of the head's output = {output_per_head.shape}")

Shape of the head's output = torch.Size([1, 12, 4, 64])


Merging the heads

In [21]:
merged_output = output_per_head.transpose(1, 2).contiguous().view(batch, time, channels)
print(f"Shape of the merged_output: {merged_output.shape}")

Shape of the merged_output: torch.Size([1, 4, 768])


In [22]:
# Pass thrught the final output
c_proj = nn.Linear(channels, channels)
final_output = c_proj(merged_output)
print(f"Shape of the final output: {final_output.shape}")

Shape of the final output: torch.Size([1, 4, 768])


| Component | Shape Transformation | Purpose |
| :--- | :--- | :--- |
| **Split Heads** | `(batch, time, channels) -> (batch, num_heads, time, head_dim)` | Prepare for parallel computation |
| **Attention** | `(batch, num_heads, time, head_dim) -> (batch, num_heads, time, head_dim)` | Each head computes context independently |
| **Merge Heads** | `(batch, num_heads, time, head_dim) -> (batch, time, channels)` | Combine the insights from all heads |
| **Final Projection** | `(batch, time, channels) -> (batch, time, channels)` | Mix the combined information |

In [23]:
class CausalSelfAttention(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    assert config.n_embed % config.n_head == 0
    self.n_head = config.n_head
    self.n_embed = config.n_embed
    self.c_attn = nn.Linear(config.n_embed, 3 * config.n_embed, bias=False)

    # we register the mask as a "buffer"
    self.register_buffer(
        "bias",
        torch.tril(torch.ones(config.block_size, config.block_size))
        .view(1, 1, config.block_size, config.block_size)
    )

    self.c_proj == nn.Linear(config.n_embed, config.n_embed) # Final Projection


  def forward(self, x):
    batch, time, channels = x.size()


    # get qkv and split into heads
    qkv = self.c_attn(x)
    query, key, value = qkv.split(self.n_embed, dim=2)

    head_dim = channels // self.n_head

    query = query.view(batch, time, self.n_head, head_dim).transpose(1, 2)
    key = key.view(batch, time, self.n_head, head_dim).transpose(1, 2)
    value = value.view(batch, time, self.n_head, head_dim).transpose(1, 2)


    # Run causel self attention
    scaled_scores = (query @ key.transpose(-2, -1)) / math.sqrt(key.size(-1))
    # Slicing the stored mask to the correct sequance length time
    scaled_scores = scaled_scores.masked_fill(self.bias[:, :, :time, :time] == 0, float('-inf'))
    attention_weights = torch.nn.functional.softmax(scaled_scores, dim=-1)

    output = attention_weights @ v

    # Merge heads and project
    output = output.transpose(1, 2).contiguous().view(batch, time, channels)
    output = self.c_proj(output)

    return output

# MLP

Attention allows tokens to interacti with eachother.
MLP processes the info for each token independently. The thinking part.

## Standard MLP architecture.
1. Expansion ('fc'): Project from n_embed up to 4* n_embed
2. Non liniarity ('gelu'): Activation function
3. Contraction ('proj'): Project back down to n_embed
4. Dropout for regulatization

nn.Linear = $input * W^{T} + b$

In [24]:
class MLP(torch.nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.fc = torch.nn.Linear(config.n_embed, 4 * config.n_embed)
    self.proj = torch.nn.Linear(4 * config.n_embed, config.n_embed)
    self.drop = torch.nn.Dropout(config.dropout)

  def forward(self, x):
    x = self.fc(x)
    x = torch.nn.functional.gelu(x)
    x = self.drop(x)
    return x

In [25]:
C_in = 2
C_out = 4

linear_layer = torch.nn.Linear(C_in, C_out)

linear_layer.weight.data = torch.tensor([
    [2., 0.],
    [-2., 1.],
    [0., 1.],
    [1., 1.]
])

linear_layer.bias.data = torch.tensor([1., 0., -1., 1.])

input_vector = torch.tensor([1., 1.])

output_vector = linear_layer(input_vector)
print(f'Input vector = {input_vector}\nOutput vector = {output_vector}')

Input vector = tensor([1., 1.])
Output vector = tensor([ 3., -1.,  0.,  3.], grad_fn=<ViewBackward0>)


## How a token goes thrugh the MLP

In [26]:
x = torch.tensor([[[1., -1.]]])

fc = torch.nn.Linear(2, 8)
torch.manual_seed(42)
fc.weight.data = torch.randn(8, 2)
fc.bias.data = torch.randn(8)
x_expanded = fc(x)

print(f"Before expansion x = {x}\nAfter Expansion x = {x_expanded}")

Before expansion x = tensor([[[ 1., -1.]]])
After Expansion x = tensor([[[ 0.9013,  3.2736,  2.4479,  2.3710, -1.2906, -0.6787, -1.1574,
          -0.5733]]], grad_fn=<ViewBackward0>)


nonliniarity

In [27]:
x_activated = torch.nn.functional.gelu(x_expanded)
print(f"x_activated shape = {x_activated.size()}\nx_activated = {x_activated}")

x_activated shape = torch.Size([1, 1, 8])
x_activated = tensor([[[ 0.7357,  3.2719,  2.4303,  2.3499, -0.1270, -0.1688, -0.1430,
          -0.1624]]], grad_fn=<GeluBackward0>)


In [28]:
proj = nn.Linear(8, 2)
proj.weight.data = torch.randn(2, 8)
proj.bias.data = torch.randn(2)
x_projected = proj(x_activated )
print(f"x_projected shape = {x_projected.size()}\nx_projected = {x_projected}")

x_projected shape = torch.Size([1, 1, 2])
x_projected = tensor([[[3.3199, 7.4928]]], grad_fn=<ViewBackward0>)


dropout. Works only durein training.

In [29]:
dropout = torch.nn.Dropout(0.1)
x_final = dropout(x_projected)
print(f"x_final  shape = {x_final .size()}\nx_final  = {x_final}")

x_final  shape = torch.Size([1, 1, 2])
x_final  = tensor([[[3.6888, 8.3253]]], grad_fn=<MulBackward0>)


## The mlp transforms the input vector while preserving it's shape!

# The skip connection.

Allows us to get rid of the vanishing gradient problem.

The express lane. The original gradient bypases all the transformations in the attention layer.

Without residuals, the netword just sees the aoutput. A blank canvas

With them the netowork alwasy sees the previous uninterrupted gradients.

It just has to learn small itterative adjustemnts

In [30]:
# Input vector
x_initial = torch.tensor([[[0.2, 0.1, 0.3, 0.4]]])

# output of self attention. The adjusten initial input
attention_output = torch.tensor([[[0.1, -0.1, 0.2, -0.3]]])

x_after_attention = x_initial + attention_output

In [31]:
print(f"Inital x = {x_initial}")
print(f"Inital x after attention = {attention_output}")
print(f"Inital x after attention and residual = {x_after_attention}")

Inital x = tensor([[[0.2000, 0.1000, 0.3000, 0.4000]]])
Inital x after attention = tensor([[[ 0.1000, -0.1000,  0.2000, -0.3000]]])
Inital x after attention and residual = tensor([[[0.3000, 0.0000, 0.5000, 0.1000]]])


# Layer Normalization

During training, when data flows thrught the network, the internal distribution of the activations of each layer is constantly changing.

The mean and var of the inputs to a given layer shift wildly from one batch to another. This is called **internal covariate shift**.

The training becomes vary hard because of this since each layer has to adapt to a new distribution of inputs from the layer before it, whitch can make training unstable, and more importantly slow.

The solution is layer normalization.

It's a technique that forces the inputs to each sublayer to have a set distribution. So it acts as a stabilizer.

For each individual token vector in (batch, time, channels) tensor, it performs theese steps:

1. calculates the mean $\mu$ and variance $\sigma^2$ acrss the $c$(embedding) dimension of that single vector.
2. Normalizes the vector. $\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$
3. Applies learnable parameters where $ y = γ * \hat{x} = β$

1.

In [32]:
x_token = torch.tensor([[[0.3, -0.2, 0.9, 0.4]]])
mean = x_token.mean(dim=-1, keepdim=True)
std = x_token.std(dim=-1, keepdim=True)

print(f"x token: {x_token}\nmean = {mean}\nstd = {std}")

x token: tensor([[[ 0.3000, -0.2000,  0.9000,  0.4000]]])
mean = tensor([[[0.3500]]])
std = tensor([[[0.4509]]])


2.

In [33]:
epsilon = 1e-5
x_hat = (x_token - mean) / torch.sqrt(std**2 + epsilon)
print(f"x gat: {x_token}\nmean = {x_hat.mean(dim=-1, keepdim=True)}\nstd = {x_hat.std(dim=-1, keepdim=True)}")

x gat: tensor([[[ 0.3000, -0.2000,  0.9000,  0.4000]]])
mean = tensor([[[0.]]])
std = tensor([[[1.0000]]])


3.

Applying the learnable parameters.

LayrNorm.weight $γ$, starts out as all ones

LayerNorm.bias $β$ starts out as all 0

These parameters are learned during training

In [34]:
gamma = torch.tensor([1.5, 1.0, 1.0, 1.0])
beta = torch.tensor([0.5, 0.0, 0.0, 0.0])


y = gamma * x_hat + beta

print(f"final output = {y}\nmean = {y.mean().item()}\nstd = {y.std().item()}")

final output = tensor([[[ 0.3337, -1.2197,  1.2197,  0.1109]]])
mean = 0.1111399382352829
std = 1.0082148313522339


| Feature | Pre-Norm (Our GPT-2 style) | Post-Norm (Original Transformer style) |
| :--- | :--- | :--- |
| **Equation** | `x + Sublayer( LayerNorm(x) )` | `LayerNorm( x + Sublayer(x) )` |
| **Stability** | Generally **more stable**. Easier to train very deep networks without fancy learning rate schedules. | Can be harder to train for very deep models. Often requires a learning rate "warm-up". |
| **Example Code** | `x = x + self.attn(self.ln_1(x))` | `x = self.ln_1(x + self.attn(x))` |

# We got all the blocks

* The communication layer (CausalSelfAttention0
* The thinking layer (MLP)
* The sexpress lane (residual/skip connections)
* the stabilizer (Layer Normalization)

In [35]:
class Block(torch.nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.ln1 = torch.nn.LayerNorm(config.n_embed) # stabilizer
    self.attention = CausalSelfAttention(config) # the communication layer
    self.ln2 = torch.nn.LayerNorm(config.n_embed) # another stabilizer
    self.mlp = MLP(config) # the thinkin layer

  def forward(self, x):

    # The outputs of the attention layers are ADDED to the original input.
    x = x + self.attention(self.ln_1(x))
    # the output of the MLP is added to the output of the first step.
    x = x + self.mlp(self.ln2(x))
    return x

The most crucial part of the block is the fact that the output shape is the same as input shape. That makes it perfectly stackable.

This allows the blocks to learn specific things, like

block one might learn basic grammar and syntax

block thwo pieces together high level concepts and ideas

...

block 12 learns deep understanding(hopefully) and learns semantic understanding.

##Width and depth

Width (multihead) Parallel processing within a layer, Analizes input from meny perspectives at once

Depth (multilayer) Sequential processing across layers, to build hierarchical understanding from simple patterns to abstrac concepts.

To code this we use toprch.nn.ModuleList. This creates a n_layer seperate instances of the Block.

The weight are not shared between theese blocks. Each block(config) crtee 12 inepenent objects.

Why model makes T predictions?

1. For training efficiency

    we want the model to predict every single position in the sequence, all at once. The 'logits' tensor fives us all the precictions we need in a single forward pass. We know thos works coz the **causal mask** ensures that prediction at position t only uses infor form tokens 0-t. This parralel approach is incredibely efficient on GPU

2. For generation (inference)

    The model produces 'logist' of shape (batch, time, vocab_size)
    
    We throw away the predictions from the first two positions

    We only use the prediction from the final position, logits [:, -1, :] to sample the next word.


| Layer | Attribute | Shape `(rows, cols)` | Role |
| :--- | :--- | :--- | :--- |
| `wte` | `self.wte.weight` | `(vocab_size, n_embd)` | **ID-to-Meaning:** Converts a token ID (row index) into a meaning vector. |
| `lm_head` | `self.lm_head.weight` | `(vocab_size, n_embd)` | **Meaning-to-ID:** Converts a meaning vector into a score for each token ID. |


theese 2 operations are inverses of eachother.

The vector that represents the word 'cat' should be the same wheather it's the input or the output.

It results in a massive parameter reduction

It also improoves performence by acting as a powerfull form of regularization. By enforcing sensible archtectural constraint, it can prevent overfitting, and lead to more stable models.

How does cross entropy work?

Calculates the loss for each of the batch*time predictions individually, and the it **averages them** to produce a single scalar 'loss'value. It keeps the loss on a consistant sale. Make the training process stable.

| Stage | What Happens | Detailed Breakdown |
| :--- | :--- | :--- |
| **1. Forward Pass** | Batch (`idx`) is fed into the model. | Model outputs a `logits` tensor of shape `(2, 4, vocab_size)`. |
| **2. Loss (Reshape)** | Prepare `logits` and `targets`. | `logits` -> `(8, vocab_size)`. `targets` -> `(8)`: `[12,8,21,6,33,9,4,15]`. |
| **3. Loss (Cross-Entropy)** | Calculate loss for each of the 8 pairs. | Individual losses: `[2.5, 3.1, 1.9, 4.2, 2.8, 3.5, 2.2, 3.8]` |
| **4. Loss (Averaging)** | <span style="color:green">Average</span> these 8 individual losses. | `final_loss = (2.5 + ... + 3.8) / 8 = 3.0` |
| **5. Backpropagation** | Calculate gradients for the batch. | `loss.backward()` is called on the single scalar value `3.0`. |
| **6. Weight Update** | The optimizer takes one step. | `optimizer.step()` updates all model weights to lower the average loss. |


## Generation Loop
1. Predict: feed the current sequence in, get a guess for the next word
2. Sample: Roll the dice on that guess and pick a new word
3. Append: Add the new word to the end of the sequence
4. back to step one, with the new sequence


In [36]:
class GPT2(torch.nn.Module):
  def __init__(self, config: GPTConfig):
    self.config = config

    # Input layers
    self.wte = torch.nn.Embedding(config.vocab_size, config.n_embed)
    self.wpe = torch.nn.Embedding(config.block_size, config.n_embed)
    self.drop = torch.nn.Dropout(config.dropout)

    # Creates a given number of independent block instances. Our data flows thrugh theese blocks sequentially
    # the core processing layers
    self.h = torch.nn.ModuleList([Block(config) for _ in range(config.n_layer)])

    # output layers.
    self.ln_f == torch.nn.LayerNorm(config.n_embed)

    # The parralel prediction space
    # Projects our internal vector space (channels dimensions) into the
    # vacab space (vocab_size dimensions)
    # Is applied in parralel and independently to every single token's vector along the T dimension
    # this mean that the toke does not make a single prediction. It makes T predictions
    #
    self.lem_head = torch.nn.Linear(config.n_embed, config.vocab_size, bias=False)
    # weight tying.
    # it's not just copying the weights, it's telling pytorch that the weight
    # matrix from the lm_head layer is the exact same object in memory as the weight matrix
    # for out token embedding layer, wte
    self.lem_head.weight = self.wte.weight

  def forward(self, idx, targets=None):
    batch, time = idx.size()

    assert T <= self.config.block_size, "Sequence length exceeds block size."

    pos = torch.arange(0, time, dtype=torch.long, device=idx.device).unsqueeze(0)

    x = self.wte(idx) + self.wpe(idx)
    x = self.drop(x)
    for block in self.h:
      x = block(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    loss = None
    if targets is not None:
      loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

    return logits, loss

  @torch.no_grad()
  def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.config.block_size:]
      logits, _ = self(idx_cond)
      logits = logits[:, -1, :] / max(temperature, 1e-8)

      if top_k is not None:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        thresh = v[:, -1].insqueeze(-1)
        logits = torch.where(logits < thresh, torch.full_like(logits, -float("inf")), logits)

      probs = torch.nn.functional.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, next_token), dim=1)
    return idx